In [6]:
import duckdb

# Join and Save File

In [7]:
# Join the K-Means and LDA features and export to file
LEFT_PATH = '../data/processed/clustered_narratives.parquet'
LDA_PATH = '../data/processed/lda_features.parquet'
OUTPUT_PATH = '../data/processed/final_unsupervised_features.parquet'

duckdb.sql(f"""
    COPY (
        SELECT
            l.*,
            r.* EXCLUDE ("Complaint ID")
        FROM '{LEFT_PATH}' AS l
        LEFT JOIN '{LDA_PATH}' AS r
            ON l."Complaint ID" = r."Complaint ID"
    )
    TO '{OUTPUT_PATH}'
    (FORMAT PARQUET)
""")

# Sanity Checks

In [8]:
# Record count of the combined file
COMBINED_PATH = '../data/processed/final_unsupervised_features.parquet'

# Full row count
row_count = duckdb.sql(f"""
    SELECT COUNT(*) AS total_rows
    FROM '{COMBINED_PATH}'
""").fetchone()[0]

print(f'Total rows: {row_count:,}')

Total rows: 398,004


In [9]:
# View the columns
duckdb.sql(f"""
    DESCRIBE SELECT *
    FROM '{COMBINED_PATH}'
""").df()

,column_name,column_type,null,key,default,extra
0,Date received,TIMESTAMP,YES,None,None,None
1,Product,VARCHAR,YES,None,None,None
2,Sub-product,VARCHAR,YES,None,None,None
3,Issue,VARCHAR,YES,None,None,None
4,Sub-issue,VARCHAR,YES,None,None,None
5,Consumer complaint narrative,VARCHAR,YES,None,None,None
6,Company public response,VARCHAR,YES,None,None,None
7,Company,VARCHAR,YES,None,None,None
8,State,VARCHAR,YES,None,None,None
9,ZIP code,VARCHAR,YES,None,None,None


In [11]:
# First 10 rows for selected columns
preview_df = duckdb.sql(f"""
    SELECT
        "Complaint ID",
        Product,
        Issue,
        dominant_topic,
        topic_4_prob,
    FROM '{COMBINED_PATH}'
    LIMIT 10
""").df()

preview_df

,Complaint ID,Product,Issue,dominant_topic,topic_4_prob
0,3442136,Credit card or prepaid card,Problem with a purchase shown on your statement,10,0.000685
1,3601853,Credit card or prepaid card,Trouble using the card,11,0.002500
2,3300820,Credit card or prepaid card,Problem with a purchase shown on your statement,11,0.057711
3,3739698,Credit card or prepaid card,Trouble using your card,11,0.203441
4,3285243,Credit card or prepaid card,"Advertising and marketing, including promotion...",1,0.113523
5,8873634,Checking or savings account,Problem with a lender or other company chargin...,17,0.341949
6,3739701,Checking or savings account,Managing an account,7,0.000055
7,7942358,Credit card,Problem with a purchase shown on your statement,17,0.290232
8,3589185,Checking or savings account,Managing an account,12,0.001250
9,3235193,Checking or savings account,Problem with a lender or other company chargin...,8,0.152911
